# RAG Pipeline Flow 
This notebook walks through every stage of the RAG pipeline in order.
Each cell maps directly to one file in the `indexing/` or `generation/` folder.

**Pipeline:**
```
INDEXING                          GENERATION
────────────────────────────      ──────────────────────────
1. Load     sample_news.json      5. Embed query
2. Chunk    split into pieces     6. Retrieve  vector search
3. Embed    text → vectors        7. Generate  LLM + context
4. Store    vectors → Qdrant      8. Response  citations
```

In [1]:
import json, os 
from dotenv import load_dotenv
from datetime import datetime # parsing date time

load_dotenv()  # reads from .env

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"
print("✓ API key loaded")

✓ API key loaded


# Indexing Pipeline

## Stage 1: Load documents
`indexing/loaders/file_loader.py`

Read raw documents and outputs a list of `documents`

In [2]:
class RawDocument:
    def __init__(self, id, title, text, url, publish_date, category="", tags=None):
        self.id = id
        self.title = title
        self.text = text
        self.url = url
        self.publish_date = publish_date
        # self.category = category
        # self.tags = tags if tags is not None else []
        
def load(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
        
    docs = []
    for doc in raw: 
        docs.append(RawDocument(
            id=doc["id"],
            title=doc["title"],
            text=doc["content"],
            url=doc["url"],
            publish_date=datetime.fromisoformat(doc["publish_date"]),
            # category=doc.get("category", ""),
            # tags=doc.get("tags", []),
        ))
        
    return docs
        
docs = load("./data/news.json")

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(f" [{doc.id}] {doc.title}")

Loaded 11 documents

 [20260203_1] 樂團「微醺開根」首發專輯 唱當代族群故事
 [20260203_2] 排灣童聲唱出部落記憶 春日國小推《春迴》專輯
 [20260203_3] 仁愛鄉梅子白蘭地風味絕佳 奪國際一星獎章
 [20260203_4] 復興推「部落社區再復興」 凝共識促永續發展
 [20260203_5] 陳義信用行動感謝 職棒生涯文物捐贈退輔會
 [20260203_6] 台北國際電玩展 設計商「赴原Saikin」展新作
 [20260203_7] 嘉義文化科技創意賽頒獎 近50組團隊參賽
 [20260203_8] 《村裡遇見熊》書展亮相 記錄人熊共生故事錄
 [20260203_9] 台北國際書展登場 原民館推講座、交流活動
 [20260203_11] 首批承領人正式取得 大南澳土地所有權狀
 [20260203_12] 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐


# Stage 2: Chunk
`indexing/chunkers/semantic_chunker.py`

Split each document into smaller pieces. 

**Why**: LLMs have content limits. Smaller chunks = more precise retrieval

Key rule: every chunk must keep `title`, `url`, `publish_date` for citation later.

In [4]:
import re

class Chunk:
    def __init__(self, chunk_id, source_id, text, title, url, publish_date):
        self.chunk_id = chunk_id
        self.source_id = source_id
        self.text = text
        self.title = title
        self.url = url
        self.publish_date = publish_date

def chunk(doc: RawDocument, max_sentences=3, overlap_sentences=1):
    """
    Sentence boundaries in Chinese naturally align with meaning boundaries. 
    Approach: 
        Chunk by sentences
    Method:
        Slide a window of max_sentences over the sentence list.
        Each step moves forward by (max_sentences - overlap_sentences),
        so the last overlap_sentences cards are always carried into the next chunk.
    """
    # Split on Chinese punctuation
    # sentences = re.split(r"(?<=[。！？])", doc.text)
    sentences = re.findall(r".+?[。！？]」?|.+?」", doc.text, re.DOTALL)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    i = 0
    step = max_sentences - overlap_sentences   # how far to advance each time
    while i < len(sentences):
        window = sentences[i : i + max_sentences]  # take up to max_sentences cards
        chunks.append("".join(window))
        i += step

    return [
        Chunk(
            source_id=doc.id,
            chunk_id=f"{doc.id}_{i}",
            text=text,
            title=doc.title,
            url=doc.url,
            publish_date=doc.publish_date,
        )
        for i, text in enumerate(chunks)
    ]

all_chunks = []
for doc in docs:
    all_chunks.extend(chunk(doc))

print(f"{len(docs)} documents → {len(all_chunks)} chunks\n")

11 documents → 52 chunks



In [5]:
# Inspect: sentences → chunks for a given news_id
news_id = "20260203_1"

doc = next(d for d in docs if d.id == news_id)

# Step 1: show how the article splits into sentences
sentences = re.findall(r".+?[。！？]」?|.+?」", doc.text, re.DOTALL)
sentences = [s.strip() for s in sentences if s.strip()]

print(f"=== Sentences ({len(sentences)} total) ===")
for i, s in enumerate(sentences):
    print(f"  [S{i}] ({len(s)} chars) {s}")

# Step 2: show how sentences are grouped into chunks
print(f"\n=== Chunks (max_sentences=3, overlap=1) ===")
example_chunks = [c for c in all_chunks if c.source_id == news_id]
for c in example_chunks:
    print(f"\n  [{c.chunk_id}] ({len(c.text)} chars)")
    print(f"  {c.text}")

=== Sentences (9 total) ===
  [S0] (82 chars) 露出靦腆笑容、輪流自我介紹，他們是成軍邁入第七年的樂團「微醺開根」，透過大提琴與木吉他的編制，演唱〈火開始的地方〉，這首作品也成為他們首張實體專輯分享會的重要主軸。
  [S1] (113 chars) 微醺開根團長 Teymu Boya（以樂） ：「這首歌其實就在講述樂團從一剛開始，然後到成軍、到現在，一路走過來，無論好的、壞的，我們都用比喻的方式，寫成像是在生活的過程，可能會遇到挫折、遇到貴人這些，我們都寫在這首歌裡面。」
  [S2] (57 chars) 團長Teymu說，團員就像圍著火堆取暖，在音樂中逐漸凝聚彼此，也透過創作，開啟年輕世代與原鄉、身分認同之間的對話。
  [S3] (46 chars) 分享會上，樂團也發表一首為平埔族群創作的歌曲，希望讓更多人透過音樂，理解不同族群的文化脈絡。
  [S4] (50 chars) 微醺開根成員 柳町 & 粉絲 Tabiliah ：「就可以去凝聚更多的力量，讓更多人知道、認識他們。
  [S5] (119 chars) 然後也用了一首噶哈巫族跟巴宰族的古謠，叫做〈aiyan〉，他其實是唱傳說故事，但是他融入了就是我們的傳統東西，就是有人不是我們族人，甚至我們自己族人知道我們自己有什麼東西都很難，但是他們願意去這樣子放在裡面，然後去理解，我覺得就是尊敬。」
  [S6] (67 chars) 至於為何成軍多年後，才正式推出首張專輯，Teymu也分享，關鍵轉折來自參加Pasiwali培訓營，在多位老師的鼓勵下，選擇走出舒適圈。
  [S7] (78 chars) 微醺開根團長 Teymu Boya（以樂） ：「也讓我們可以往更專業的地方走，也給我們很多的信心，讓我們相信自己是有能力，跟已經準備好要來做這樣子的挑戰。」
  [S8] (43 chars) 透過親自創作的樂、詞、曲，微醺開根唱出成長的軌跡，也用作品，為族群記憶留下當代的聲音。

=== Chunks (max_sentences=3, overlap=1) ===

  [20260203_1_0] (252 chars)
  露出靦腆笑容、輪流自我介紹，他們是成軍邁入第七年的樂團「微醺開根」，透過大提琴與木吉他的編制，演唱〈火開始的地方〉，這首作品也成為他們

# Stage 3: Embed
`indexing/embedders/bge_m3.py`

Convert each chunk's text into a vector representation. *Why*: vectors capture semantic meaning and similar meaning means similar vector.

In [ ]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding # Can also try jieba for Chinese text segmentation

# Dense Vector Embedder Choice: "BAAI/bge-m3" / "paraphrase-multilingual-MiniLM-L12-v2"
dense_embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# Sparse Vector Choice: "Qdrant/bm25"
sparse_embedder = SparseTextEmbedding(model_name="Qdrant/bm25")  # Keyword matching

texts = [chunk.text for chunk in all_chunks]

# Dense: semantic embedding (fixed-size vectors)
dense_vectors = dense_embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
print(type(dense_vectors))

# Sparse: keyword embedding (variable-size vectors, mostly zeros)
sparse_vectors = list(sparse_embedder.embed(texts))
print(type(sparse_vectors))

print(f"\nDense:  {len(dense_vectors)} chunks, dimension={len(dense_vectors[0])}")
print(f"Sparse: {len(sparse_vectors)} chunks, example nnz={len(sparse_vectors[0].indices)} non-zero values")

/home/hanyusu/newsLLM-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4007.31it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:00<00:00,  7.60it/s]

<class 'numpy.ndarray'>
<class 'list'>

Dense:  52 chunks, dimension=384
Sparse: 52 chunks, example nnz=29 non-zero values


# Stage 4: Store
`indexing/vector_stores/qdrant_store.py`

Save vectors + chunks metadata into Qdrant.

In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, SparseVector, SparseVectorParams, SparseIndexParams, Modifier

# :memory: = no Docker needed. Swap to host="localhost" when using Docker
client = QdrantClient(":memory:")

COLLECTION = "hybrid_search"
DENSE_VECTOR_DIM = len(dense_vectors[0])

# Create a collection:
# A collection is a named set of points (vectors with a payload) among which you can search
client.create_collection(
    collection_name=COLLECTION,
    
    # Dense Vector captures semantic meaning
    vectors_config={
        "dense": VectorParams(size=DENSE_VECTOR_DIM, distance=Distance.COSINE)
    },
    
    # Sparse Vector captures keyword 
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False),
            modifier=Modifier.IDF # (Inverse Document Frequency) words that appear frequently in every chunk get lower weight and rare words get higher weights
        )
    }
)


points = [
    # Each chunk is stored as a record in the database (id + vector + metadata)
    PointStruct(
        id=i,
        vector={
            "dense": dense_vectors[i].tolist(),
            "sparse": SparseVector(                         # Most of positions are zeros 
                indices=sparse_vectors[i].indices.tolist(), # WHICH positions have values
                values=sparse_vectors[i].values.tolist(),   # WHAT those values are
            )
        },
        payload={ # metadata for citation
            "chunk_id": chunk.chunk_id,
            "source_id": chunk.source_id,
            "text":      chunk.text,
            "title":     chunk.title,
            "url":       chunk.url,
            "publish_date": chunk.publish_date.isoformat(),
        }
    )
    for i, chunk in enumerate(all_chunks) # Iterate through a list of chunk object
]

# Update or insert a new point to a collection
client.upsert(collection_name=COLLECTION, points=points)

info = client.get_collection(COLLECTION)
print(f"✓ Collection '{COLLECTION}' ready")
print(f"  {info.points_count} vectors stored, dimension={DENSE_VECTOR_DIM}")

✓ Collection 'hybrid_search' ready
  52 vectors stored, dimension=384


# Generation Pipeline 

## Stage 5: Embed Query
Use the **same model** as stage 3 to embed the user's question. 

**Why**: if we use a different model here, the vectors live in different spaces and retrieval spaces and retrieval silently breaks.

In [12]:
query = "2026年台灣的詐騙事件有什麼？"

# Same embedder object --- same model guaranteed
query_dense_vector = dense_embedder.encode(query, normalize_embeddings=True).tolist()
# list(...) — forces the generator to compute and collect all results:
query_sparse_vector = list(sparse_embedder.embed([query]))[0] # [0] — extract the first (and only) query result 

print(f"Query: {query}")
print(f"Dense dim:  {len(query_dense_vector)}")
print(query_sparse_vector.indices)   # which vocabulary position
print(query_sparse_vector.values)    # its weight
print(f"Sparse nnz: {len(query_sparse_vector.indices)} non-zero values")

Query: 2026年台灣的詐騙事件有什麼？
Dense dim:  384
[786509639]
[1.68774348]
Sparse nnz: 1 non-zero values


# Stage 6: Retrieve 
`generation/retrievers/hybrid_retriever.py`

Search Qdrant for the chunks most similar to the query vector.

Print Scores so we can visualize what model finds relevant.

In [25]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion

class RetrievedChunk:
    def __init__(self, chunk, score):
        self.chunk = chunk
        self.score = score


def retrieve(query_dense_vector, query_sparse_vector, top_k=3):
    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            # Step 1a: dense search — finds semantically similar chunks
            Prefetch(
                query=query_dense_vector,
                using="dense",
                limit=top_k * 2,   # fetch more candidates than needed
            ),
            # Step 1b: sparse search — finds keyword matching chunks
            Prefetch(
                query=SparseVector(
                    indices=query_sparse_vector.indices.tolist(),
                    values=query_sparse_vector.values.tolist(),
                ),
                using="sparse",
                limit=top_k * 2,
            ),
        ],
        # Step 2: Reciprical Rank Fusion (RRF): merges both ranked lists into one final ranking
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    ).points

    return [
        RetrievedChunk(
            chunk=Chunk(
                chunk_id=r.payload["chunk_id"],
                source_id=r.payload["source_id"],
                text=r.payload["text"],
                title=r.payload["title"],
                url=r.payload["url"],
                publish_date=datetime.fromisoformat(r.payload["publish_date"]),
            ),
            score=r.score,
        )
        for r in results
    ]


retrieved = retrieve(query_dense_vector, query_sparse_vector, top_k=3)

print(f"Top {len(retrieved)} results for: '{query}'\n")
for i, r in enumerate(retrieved):
    print(f"  [{i+1}] score={r.score:.4f} | {r.chunk.title}")
    print(f"       {r.chunk.text}")

Top 3 results for: '2026年台灣的詐騙事件有什麼？'

  [1] score=0.5000 | 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐
       花蓮刑大大隊長 劉宗仁：「我們之後把資料把它做比對之後，我發現有3個地方是不一樣的，第一個是它的流水編號是不一樣的，第2個它的指定用途是不一樣的，第3個它的收據的編號是不一樣的，所以這3點經過比對之後，我們確定它是偽造的，我們是確定說它是用這樣子一個，假的收據來取信被害人，營造他的那個樂善好施的一個人設，所以我們知道說已經有，很多這種方式在進行詐騙，如果民眾有這種詐騙的時候，來跟我們警方聯繫，我們來偵辦。」花蓮縣府新聞科長 李冠霆：「縣府最近有接到民眾的反映，那有民眾收到有人持那個偽造的，捐款收據，然後來取信他人，那經我們縣府查證後，那確定這些收據並不是，花蓮縣政府所開立，而且相關的捐款證明也不存在，花蓮縣政府在此也呼籲大家，提高警覺，然後並洽我們，花蓮縣政府社會處來做確認，避免善心被不法人士所利用。」社會處進一步指出，經檢視這些收據，發現流水編號、指定用途名稱以及字體格式，都與正式版本明顯不符，且有多處修改痕跡，不排除不肖人士意圖利用假收據博取信任，誘導民眾交付金錢。
  [2] score=0.3333 | 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐
       社會處進一步指出，經檢視這些收據，發現流水編號、指定用途名稱以及字體格式，都與正式版本明顯不符，且有多處修改痕跡，不排除不肖人士意圖利用假收據博取信任，誘導民眾交付金錢。花蓮刑大大隊長 劉宗仁：「本局已經介入了解處理，研判相關的手法是以假捐款收據，來取信被害人，在誘導被害人來投資，或進行其他的詐騙行為，目前我們經清查之後，尚未有接獲正式的報案，我們會持續來，清查跟釐清，是否有相關的財務損失。」縣府強調，目前已蒐集相關事證，將依法處理，維護公益捐款秩序與民眾權益，也呼籲民眾切勿輕信來路不明的捐款資訊，更不要隨意提供個人或財務資料。
  [3] score=0.2500 | 台北國際電玩展 設計商「赴原Saikin」展新作
       細心向民眾介紹今年最新發行的桌遊，仔細一看，遊戲設計充滿原住民族文化傳統元素。桌遊設計商「赴原Saikin」，致力於透過遊戲化方式，傳遞台灣各原住民族及其他族群的多元文化。今年，他們再次藉由台北國際電玩展

# Stage 7: Generate
`generation/llms/openai_llm.py`

Pass retrieved chunks as context to the LLM.

Evidence-first prompt: show sources first, then ask the question

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def build_prompt(query: str, chunks: list[RetrievedChunk]) -> str:
    evidence = "\n\n".join(
        f"[{i+1}] 標題：{c.chunk.title}\n"
        f"    來源：{c.chunk.url}\n"
        f"    內容：{c.chunk.text}\n"
        f".   報導時間: {c.chunk.publish_date}"
        for i, c in enumerate(chunks)
    )
    return f"""你是一個可信新聞助理。請根據以下參考資料回答問題。
回答時標註來源URL連結，若資料不足請說明無法確認。

=== 參考資料 ===
{evidence}

=== 問題 ===
{query}"""


def generate(query: str, chunks: list[RetrievedChunk]) -> str:
    prompt = build_prompt(query, chunks)
    print(prompt)
    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content

# Ask LLM to generate the response based on the retrieved context
answer = generate(query, retrieved)
print("\n\nLLM generated response:")
print(answer)

你是一個可信新聞助理。請根據以下參考資料回答問題。
回答時標註來源URL連結，若資料不足請說明無法確認。

=== 參考資料 ===
[1] 標題：冒用縣府名義詐捐 花蓮縣府急澄清籲防詐
    來源：https://news.ipcf.org.tw/194836
    內容：花蓮刑大大隊長 劉宗仁：「我們之後把資料把它做比對之後，我發現有3個地方是不一樣的，第一個是它的流水編號是不一樣的，第2個它的指定用途是不一樣的，第3個它的收據的編號是不一樣的，所以這3點經過比對之後，我們確定它是偽造的，我們是確定說它是用這樣子一個，假的收據來取信被害人，營造他的那個樂善好施的一個人設，所以我們知道說已經有，很多這種方式在進行詐騙，如果民眾有這種詐騙的時候，來跟我們警方聯繫，我們來偵辦。」花蓮縣府新聞科長 李冠霆：「縣府最近有接到民眾的反映，那有民眾收到有人持那個偽造的，捐款收據，然後來取信他人，那經我們縣府查證後，那確定這些收據並不是，花蓮縣政府所開立，而且相關的捐款證明也不存在，花蓮縣政府在此也呼籲大家，提高警覺，然後並洽我們，花蓮縣政府社會處來做確認，避免善心被不法人士所利用。」社會處進一步指出，經檢視這些收據，發現流水編號、指定用途名稱以及字體格式，都與正式版本明顯不符，且有多處修改痕跡，不排除不肖人士意圖利用假收據博取信任，誘導民眾交付金錢。
.   報導時間: 2026-02-03 19:00:00+08:00

[2] 標題：冒用縣府名義詐捐 花蓮縣府急澄清籲防詐
    來源：https://news.ipcf.org.tw/194836
    內容：社會處進一步指出，經檢視這些收據，發現流水編號、指定用途名稱以及字體格式，都與正式版本明顯不符，且有多處修改痕跡，不排除不肖人士意圖利用假收據博取信任，誘導民眾交付金錢。花蓮刑大大隊長 劉宗仁：「本局已經介入了解處理，研判相關的手法是以假捐款收據，來取信被害人，在誘導被害人來投資，或進行其他的詐騙行為，目前我們經清查之後，尚未有接獲正式的報案，我們會持續來，清查跟釐清，是否有相關的財務損失。」縣府強調，目前已蒐集相關事證，將依法處理，維護公益捐款秩序與民眾權益，也呼籲民眾切勿輕信來路不明的捐款資訊，更不要隨意提供個人或財務資料。
.   報導時間: 2026-02-03 19:00:00+08:00

[3]